# One RL action step (quantum policy)

Sample an action from a policy `relu(W·x) → state` measured in a fixed basis. Same trace, two routes.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** sample an action from a policy distribution. **Classical:** tabular Q-learning / softmax policy. **Quantum:** angle-encode policy state, projective measurement gives the action index.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.ops import linalg, sample
from uniqx.ops.primitives.nn import relu

n = 4
rng = np.random.default_rng(20)
W = rng.standard_normal((n, n)) * 0.3
x = np.array([0.5, 0.0, -0.2, 0.1])
n_samples = 100
O_id = np.eye(n)

@trace
def prog(weight, vec, observable):
    h = relu(linalg.matmul(weight, vec))
    return sample(observable, h, n_samples=n_samples)

module = prog(W.tolist(), x.tolist(), O_id.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
